# Take the dilution problem to Ax

The dilution problem from [Optimization Live](https://github.com/benderill/agent-O): add water to 1 ml of blue dye until the color matches a pale target. The objective is the color difference ΔE, minimized, with a budget of 8 mixes.

The script was generated with [Honegumi](https://honegumi.readthedocs.io/en/latest/) (single objective, default model, no constraints) and adapted. Every change is marked `# MOD:`. The unedited Honegumi output is in `ax/generated/dilution.honegumi.py` in the repository.

The "experiment" here is the same simulated color model the website uses, so you can run it anywhere. For a real lab, replace `measure_delta_e(water)` with your measurement.

Run all cells. The first one installs the versions this notebook was tested with.

In [1]:
%pip install ax-platform==0.4.3 "pandas<3"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Generated by Honegumi (https://arxiv.org/abs/2502.06815)
import numpy as np
from ax.service.ax_client import AxClient, ObjectiveProperties

# MOD: the simulated dilution lab from Optimization Live replaces Honegumi's test function.
# Same model as the browser (src/problems/color-model.js), not real chemistry.
import math

TARGET_WATER_ML = 11.0


def dilution_rgb(water):
    """Display color (r, g, b in 0..1) of 1 ml of blue dye diluted with `water` ml of water."""
    c = 1.0 / (1.0 + water)  # dye share of the total volume
    return (math.exp(-2.6 * c), math.exp(-1.5 * c), math.exp(-0.35 * c))


def _srgb_to_linear(v):
    return v / 12.92 if v <= 0.04045 else ((v + 0.055) / 1.055) ** 2.4


def lab_of(rgb):
    """sRGB (0..1) to CIELAB, D65 white point."""
    r, g, b = (_srgb_to_linear(v) for v in rgb)
    X = 0.4124564 * r + 0.3575761 * g + 0.1804375 * b
    Y = 0.2126729 * r + 0.7151522 * g + 0.072175 * b
    Z = 0.0193339 * r + 0.119192 * g + 0.9503041 * b

    def f(t):
        return t ** (1.0 / 3.0) if t > 0.008856 else 7.787 * t + 16.0 / 116.0

    fx, fy, fz = f(X / 0.95047), f(Y), f(Z / 1.08883)
    return (116.0 * fy - 16.0, 500.0 * (fx - fy), 200.0 * (fy - fz))


def delta_e76(lab_a, lab_b):
    """Perceived color difference. 0 is identical, below about 2 is hard to tell apart."""
    return math.sqrt(sum((p - q) ** 2 for p, q in zip(lab_a, lab_b)))


TARGET_LAB = lab_of(dilution_rgb(TARGET_WATER_ML))


def measure_delta_e(water):
    """The black box: mix 1 ml of dye with `water` ml of water and return delta E to the target."""
    return delta_e76(lab_of(dilution_rgb(water)), TARGET_LAB)


obj1_name = "delta_e"  # MOD: color difference to the target, lower is better
N_TRIALS = 8  # MOD: the mix budget, as in the browser


ax_client = AxClient(random_seed=1, verbose_logging=False)  # MOD: fixed seed, quieter logs

ax_client.create_experiment(
    parameters=[
        # MOD: one input, the water added to 1 ml of dye, in ml
        {"name": "water", "type": "range", "bounds": [0.0, 30.0]},
    ],
    objectives={
        obj1_name: ObjectiveProperties(minimize=True),
    },
)


for i in range(N_TRIALS):  # MOD: budget

    parameterization, trial_index = ax_client.get_next_trial()

    # extract parameters
    water = parameterization["water"]  # MOD: our parameter

    results = measure_delta_e(water)  # MOD: run the (simulated) experiment
    print(f"Mix {trial_index + 1}: 1 ml dye + {water:.1f} ml water -> delta E {results:.3f}")  # MOD
    ax_client.complete_trial(trial_index=trial_index, raw_data=results)

# MOD: report the best mix that was actually measured
df = ax_client.get_trials_data_frame()
best = df.loc[df[obj1_name].idxmin()]
print(f"Best mix: 1 ml dye + {best['water']:.2f} ml water, delta E {best[obj1_name]:.3f}" f" (target: {TARGET_WATER_ML} ml, delta E 0)")


C:\Users\bdas\codes\optimization-live-repo\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[WARNING 11-21 17:07:45] ax.service.ax_client: Random seed set to 1. Note that this setting only affects the Sobol quasi-random generator and BoTorch-powered Bayesian optimization models. For the latter models, setting random seed to the same number for two optimizations will make the generated trials similar, but not exactly the same, and over time the trials will diverge more.


[INFO 11-21 17:07:45] ax.service.utils.instantiation: Inferred value type of ParameterType.FLOAT for parameter water. If that is not the expected value type, you can explicitly specify 'value_type' ('int', 'float', 'bool' or 'str') in parameter dict.


[INFO 11-21 17:07:45] ax.service.utils.instantiation: Created search space: SearchSpace(parameters=[RangeParameter(name='water', parameter_type=FLOAT, range=[0.0, 30.0])], parameter_constraints=[]).


[INFO 11-21 17:07:45] ax.modelbridge.dispatch_utils: Using Models.BOTORCH_MODULAR since there is at least one ordered parameter and there are no unordered categorical parameters.


[INFO 11-21 17:07:45] ax.modelbridge.dispatch_utils: Calculating the number of remaining initialization trials based on num_initialization_trials=None max_initialization_trials=None num_tunable_parameters=1 num_trials=None use_batch_trials=False


... (6 outputs left out)

Mix 6: 1 ml dye + 20.1 ml water -> delta E 7.157


Mix 7: 1 ml dye + 14.3 ml water -> delta E 3.515


[WARNING 11-21 17:07:48] ax.service.utils.report_utils: Column reason missing for all trials. Not appending column.


Mix 8: 1 ml dye + 11.8 ml water -> delta E 0.991
Best mix: 1 ml dye + 11.79 ml water, delta E 0.991 (target: 11.0 ml, delta E 0)


Built on [Honegumi](https://github.com/sgbaird/honegumi) (MIT license; Baird, Falkowski and Sparks, *npj Computational Materials*, 2026, doi:10.1038/s41524-026-02156-0) and [Ax](https://ax.dev/) (MIT license).